# Load necessary packages

In [1]:
import pandas as pd
import requests
import re
from ast import literal_eval
from Bio import SeqIO
from Bio.KEGG import REST
from Bio.KEGG import Enzyme

In [ ]:
# may be needed on some machines to make REST API calls
'''
import ssl
import certifi
print(certifi.where())
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())
'''

# Build data frames containing KOs for vitamin synthesis pathways

Pathways, genes, and KOs used here were curated by:

Hessler, T., Huddy, R. J., Sachdeva, R., Lei, S., Harrison, S. T. L., Diamond, S., & Banfield, J. F. (2023). Vitamin interdependencies predicted by metagenomics-informed network analyses and validated in microbial community microcosms. Nature communications, 14(1), 4768. https://doi.org/10.1038/s41467-023-40360-4

Gregor, R., Vercelli, G. T., Szabo, R. E., Gralka, M., Reynolds, R. C., Qu, E. B., Levine, N. M., & Cordero, O. X. (2025). Vitamin auxotrophies shape microbial community assembly on model marine particles. The ISME journal, 19(1), wraf184. https://doi.org/10.1093/ismejo/wraf184

Additionally, a list of keywords is created for each vitamin, indicating phrases that may be found in descriptions of annotated genes. These may be useful in case a predicted gene was not annotated with any KO or EC numbers.

In [2]:
# building reference dataframes with KO requirements

# Hessler pantothenate
pantothenate = pd.DataFrame({
  "description": ["Beta-Alanine for Pantothenate biosynthesis (step 1.1)",
                  "3M-2O > 2D for Panotthenate (step 1.2)",
                  "Pantoanate > Pantothenate (step 2 of 2)"],
  "req": [1, 2, 1],
  "altreq": [1, 2, 1],
  "KO": [["K01593", "K00128", "K07250", "K01579", "K01579"],
         ["K00826", "K00606", "K00077"],
         ["K01918"]]
})
pantothenate_keywords = ["beta-alanine", "3M-2O", "pantoanate", "pantothenate"]

# Hessler thiamine
thiamine = pd.DataFrame({
  "description": ["Kinases (thiD)",
                  "Kinases (other)",
                  "Pyrimidine",
                  "Thiazole",
                  "Linking",
                  "thiG"],
  "req": [1, 2, 1, 7, 1, 1],
  "altreq": [1, 2, 0, 7, 1, 1],
  "KO": [["K14153", "K00877", "K00941"],
         ["K00878", "K00946"],
         ["K03147", "K18278"],
         ["K03150", "K03153", "K03148", "K03151", "K01662", "K03149", "K03154", "K04487"],
         ["K14153", "K00788"],
         ["K03149"]]
})
# Gregor thiamine
thiamine_alt = pd.DataFrame({
    "description": ["thiC",
                    "thiG4",
                    "thiEN"],
    "req":[1,1,1],
    "KO": [["K03147"],
           ["K03149", "K03146", "K22699"],
           ["K00788","K00949", "K14153"]]
})
thiamine_keywords = ["thiD", "thiG", "pyrimidine", "thia", "thiazole", "thiC", "thi4", "thiE", "thiN"]

# Hessler riboflavin
riboflavin = pd.DataFrame({
    "description": ["1", "2", "3", "4", "5", "6", "7"],
    # note that presence of 5 out of 7 is considered sufficient to classify organism as a riboflavin producer
    "req": [1,1,1,1,1,1,1],
    "KO": [["K02858", "K14652"],
["K22912", "K20861", "K20862", "K21063", "K21064"],
["K01497"],
["K11752"],
["K00082"],
["K00794"],
["K00793"]]
})
riboflavin_keywords = ["riboflavin"]

# Hessler biotin
# "K19563" is not included in the Hessler paper, but corresponds to bioA for an alternative pathway used by B. subtilis.
biotin = pd.DataFrame({
    "description": ["bioCIW", "bioABDF"],
    "req": [1,3],
    "altreq": [1,3],
    "KO":[["K02169", "K16593", "K01906"],
          ["K00652", "K00833", "K01935", "K01012", "K19563"]]
})

# Gregor biotin
biotin_alt = pd.DataFrame({
    "description": ["bioF", "bioA", "bioD", "bioB"],
    "req": [1,1,1,1],
    "KO": [["K00652"],["K00833","K19563"],["K01935"],["K01012"]]
})
biotin_keywords = ["bioC", "bioI", "bioW", "bioA", "bioB", "bioD", "bioF", "biotin", "8-amino-7-oxononanoate"]

# Hessler niacin
niacin = pd.DataFrame({
    "description": ["nadC", "nadD", "nadE", "nadXB", "nadA"],
    "req": [1, 1, 1, 1, 1],
    "KO": [["K00767"],["K00969"],["K01916","K01950"],["K06989", "K00278"],["K03517"]]
})

# Gregor niacin
niacin_alt = pd.DataFrame({
    "description": ["nadC", "nadD", "nadE", "kmo"],
    "req": [1, 1, 1, 1],
    "KO": [["K00767"],["K00969"],["K01916","K01950"],["K00486"]]
})
niacin_keywords = ["QNS", "QPRT", "ASPDH", "nadC", "nadD", "nadE", "nadX", "nadB", "nadA", "kmo", "niacin", "nicotin", "NAD"]

# Define functions to collect KOs from Prokka-annotated genomes and check for KOs of interest

`ec_to_product_KO()` queries the KEGG API with an EC input (as a string. e.g.: `"ec:5.4.2.2"`) to get the corresponding KO and description, returned as lists.

`parse_genome()` applies `ec_to_product_KO()` to all ECs in a Prokka-annotated genome. The input should be a SeqRecord iterator object obtained by applying `SeqIO.parse()` to a gbff file, e.g: `SeqIO.parse("my_input_file.gbff", "genbank")`. The output will be a dataframe with columns of lists: the `product` and `gene_synonyms` columns can be searched for relevant keywords, and the `KO` column can be searched for relevant KOs. Because `ec_to_product_KO()` makes REST API calls one at a time for each EC, `parse_genome` can take a long time to run (~30+ minutes). There should be a way to query multiple ECs at once (e.g. https://rest.kegg.jp/link/ko/ec:5.4.2.2+ec:1.1.1.1+ec:2.5.1.17), and that would be faster, but one-at-a-time was sufficient for our purposes.

`cross_check()` takes 1) the reference dataframe, 2) the keyword list, and 3) the parse_genome() output dataframe, and searches the output dataframe for the relevant KOs and keywords, which are then printed to the console.

In [3]:
# function to convert EC IDs to KOs
def ec_to_product_KO(ec_id):
  ec_id = ec_id.lower()
  try:
    ec_file = REST.kegg_get(ec_id).read().rstrip().split("\n")
  except Exception as e:
    print("Rest API Error", ec_id, e)
    return [], []
  start = False
  product, KO = [], []  
  for line in ec_file:
    if line.startswith("ORTHOLOGY"):
      start = True
    if start:
      tokens = re.split(r"\s{2,}", line)
      if tokens[0] in ("ORTHOLOGY", ""):
        product.append(tokens[2])
        KO.append(tokens[1])
      else:
        break
  return(product, KO)

# function to extract all KOs (inlcuding ECs converted to KOs) from Prokka-annotated gbff files
def parse_genome(input):
  output = []
  for scaffold in input:
    for feature in scaffold.features:
      if feature.type == "gene" and "db_xref" in feature.qualifiers:
        KOs = []
        products = [feature.qualifiers['product'][0]]
        for token in feature.qualifiers['db_xref']:
          if token.startswith('KO'):
            KOs.append(token[3:])
          else:
            a = ec_to_product_KO(token)
            KOs.extend(a[1])
            products.extend(a[0])
        my_gene = [products, KOs]
        if "gene_synonym" in feature.qualifiers:
          my_gene.append(list(feature.qualifiers['gene_synonym']))
        else:
          my_gene.append([])
        output.append(my_gene)
  return output

# function to compare KOs against reference databases for pathway
def cross_check(ref_df, keywords, Ca_df):
  genome_KOs = set(Ca_df.explode("KO")["KO"].tolist())
  tracker = []
  for i in range(0, len(ref_df.index)):
      step_num = i + 1
      step_name = ref_df['description'][i]
      req = ref_df['req'][i]
      has = set(ref_df['KO'][i]).intersection(genome_KOs)
      print(f'Step {step_num}, {step_name}: {len(has)} of {req} KOs found')
      if len(has) > 0:
        print(has)
      if len(has) >= req:
        tracker.append(True)
      else:
        tracker.append(False)
      print()
  if not all(tracker):
      words = []
      genome_words = Ca_df.explode("product")["product"].tolist() + Ca_df.explode("gene_synonym")["gene_synonym"].tolist()
      genome_string = '\t'.join(str(x).lower() for x in genome_words)
      for word in keywords:
        if word.lower() in genome_string:
          words.append(word)
      if len(words) > 0:
        print('Found keywords', words)
      else:
        print('No keywords found')

# Template: Check an annotated genome for vitamin auxotrophy

In [ ]:
# parse the genome and collect all KOs

'''!!!
parse_genome() takes a while to run (30-60 min); recommended to run one genome at a time and download the output before proceeding.
Will print error statement for any EC where kegg_get() fails. These ECs can be checked manually.
!!!'''

# replace "my_input_file.gbff" with your GBFF file, and "my_output_file.csv" with your desired output filename
my_output = parse_genome(SeqIO.parse("my_input_file.gbff", "genbank"))
my_df = pd.DataFrame(my_output, columns = ['product', 'KO', 'gene_synonym'])
my_df.to_csv('my_output_file.csv')
my_df.head()

In [ ]:
# use cross_check() to find for vitamin biosynthesis genes

# my_df = pd.read_csv('Ca19_output_file.csv') # reload df from previous step if needed 
# my_df['KO'] = Ca19_df['KO'].apply(literal_eval) # if reloading df from csv, turn string back into list

print('PANTOTHENATE (Hessler)')
cross_check(pantothenate, pantothenate_keywords, my_df)
print('\nPANTOTHENATE (Hessler)')
cross_check(biotin, biotin_keywords, my_df)
print('\nBIOTIN (Gregor)')
cross_check(biotin_alt, biotin_keywords, my_df)
print('\nNIACIN (Gregor)')
cross_check(niacin, niacin_keywords, my_df)
print('\nNIACIN (Gregor alt.)')
cross_check(niacin_alt, niacin_keywords, my_df)
print('\nRIBOFLAVIN (Hessler)')
cross_check(riboflavin, riboflavin_keywords, my_df)
print('\nTHIAMINE (Hessler)')
cross_check(thiamine, thiamine_keywords, my_df)
print('\nTHIAMINE (Gregor)')
cross_check(thiamine_alt, thiamine_keywords, my_df)

# Code used to check our dependents and producers

In [4]:
Ca19_output = parse_genome(SeqIO.parse("Ca19_input_file.gbff", "genbank"))
Ca19_df = pd.DataFrame(Ca19_output, columns = ['product', 'KO', 'gene_synonym'])
Ca19_df.to_csv('Ca19_output_file.csv')
Ca19_df.head()

,product,KO,gene_synonym
0,"[Serine--tRNA ligase, seryl-tRNA synthetase]","[K01875, K01875]","[serS, serS]"
1,"[5'-nucleotidase SurE, 5'-nucleotidase, 5'-nuc...","[K03787, K01081, K02566, K03787, K08693, K1175...","[surE, surE]"
2,"[Protein-L-isoaspartate O-methyltransferase, p...","[K00573, K00573]","[pcm_1, pcm_1]"
3,[hypothetical protein],[K06194],[]
4,[hypothetical protein],[K06923],[]


In [5]:
Ca24A_output = parse_genome(SeqIO.parse("Ca24A_input_file.gbff", "genbank"))
Ca24A_df = pd.DataFrame(Ca24A_output, columns = ['product', 'KO', 'gene_synonym'])
Ca24A_df.to_csv('Ca24A_output_file.csv')
Ca24A_df.head()

,product,KO,gene_synonym
0,[hypothetical protein],[K21470],[]
1,[hypothetical protein],[K06884],[]
2,[Protein-glutamate methylesterase/protein-glut...,"[K03412, K03411, K03412, K26021]","[cheB_1, cheB_1]"
3,[hypothetical protein],[K13924],[]
4,[hypothetical protein],[K00441],[]


In [6]:
Pr24B_output = parse_genome(SeqIO.parse("Pr24B_input_file.gbff", "genbank"))
Pr24B_df = pd.DataFrame(Pr24B_output, columns = ['product', 'KO', 'gene_synonym'])
Pr24B_df.to_csv('Pr24B_output_file.csv')
Pr24B_df.head()

,product,KO,gene_synonym
0,[Protease HtpX],[K06013],"[htpX_1, htpX_1]"
1,"[Isocitrate lyase, isocitrate/methylisocitrate...","[K01637, K01637]","[aceA, aceA]"
2,[Dipeptidyl-peptidase 5],[K01303],"[dpp5, dpp5]"
3,[3-oxoacyl-[acyl-carrier-protein] reductase Fa...,"[K00059, K00059]","[fabG_1, fabG_1]"
4,[ATP-dependent helicase/deoxyribonuclease subu...,[K16899],"[addB, addB]"


In [7]:
Me12_output = parse_genome(SeqIO.parse("Me12_input_file.gbff", "genbank"))
Me12_df = pd.DataFrame(Me12_output, columns = ['product', 'KO', 'gene_synonym'])
Me12_df.to_csv('Me12_output_file.csv')
Me12_df.head()

,product,KO,gene_synonym
0,[4-methylaminobutanoate oxidase (formaldehyde-...,"[K00315, K19191]","[abo_1, abo_1]"
1,"[Dimethlysulfonioproprionate lyase DddP, dimet...","[K16953, K28067, K28068, K28069]","[dddP_1, dddP_1]"
2,"[Methionine synthase, 5-methyltetrahydrofolate...","[K00548, K24042]","[metH_1, metH_1]"
3,[hypothetical protein],[K07114],[]
4,[ECF RNA polymerase sigma factor SigW],[K03088],"[sigW_1, sigW_1]"


In [8]:
# Ca19_df = pd.read_csv('Ca19_output_file.csv') # reload df from previous step if needed 
# Ca19_df['KO'] = Ca19_df['KO'].apply(literal_eval) # if reloading df from csv, turn string back into list

print('BIOTIN (Hessler)')
cross_check(biotin, biotin_keywords, Ca19_df)
print('\nBIOTIN (Gregor)')
cross_check(biotin_alt, biotin_keywords, Ca19_df)
print('\nNIACIN (Gregor)')
cross_check(niacin, niacin_keywords, Ca19_df)
print('\nNIACIN (Gregor alt.)')
cross_check(niacin_alt, niacin_keywords, Ca19_df)
print('\nRIBOFLAVIN (Hessler)')
cross_check(riboflavin, riboflavin_keywords, Ca19_df)
print('\nTHIAMINE (Hessler)')
cross_check(thiamine, thiamine_keywords, Ca19_df)
print('\nTHIAMINE (Gregor)')
cross_check(thiamine_alt, thiamine_keywords, Ca19_df)

BIOTIN (Hessler)
Step 1, bioCIW: 1 of 1 KOs found
{'K02169'}

Step 2, bioABDF: 3 of 3 KOs found
{'K00652', 'K01935', 'K01012'}


BIOTIN (Gregor)
Step 1, bioF: 1 of 1 KOs found
{'K00652'}

Step 2, bioA: 0 of 1 KOs found

Step 3, bioD: 1 of 1 KOs found
{'K01935'}

Step 4, bioB: 1 of 1 KOs found
{'K01012'}

Found keywords ['bioC', 'bioB', 'bioD', 'bioF', 'biotin', '8-amino-7-oxononanoate']

NIACIN (Gregor)
Step 1, nadC: 1 of 1 KOs found
{'K00767'}

Step 2, nadD: 1 of 1 KOs found
{'K00969'}

Step 3, nadE: 1 of 1 KOs found
{'K01950'}

Step 4, nadXB: 1 of 1 KOs found
{'K00278'}

Step 5, nadA: 1 of 1 KOs found
{'K03517'}


NIACIN (Gregor alt.)
Step 1, nadC: 1 of 1 KOs found
{'K00767'}

Step 2, nadD: 1 of 1 KOs found
{'K00969'}

Step 3, nadE: 1 of 1 KOs found
{'K01950'}

Step 4, kmo: 0 of 1 KOs found

Found keywords ['nadC', 'nadD', 'nadE', 'nadB', 'nadA', 'nicotin', 'NAD']

RIBOFLAVIN (Hessler)
Step 1, 1: 0 of 1 KOs found

Step 2, 2: 0 of 1 KOs found

Step 3, 3: 0 of 1 KOs found

Step 4, 4: 0

In [9]:
# Ca24A_df = pd.read_csv('Ca24A_output_file.csv') # reload df from previous step if needed 
# Ca24A_df['KO'] = Ca24A_df['KO'].apply(literal_eval) # if reloading df from csv, turn string back into list

print('BIOTIN (Hessler)')
cross_check(biotin, biotin_keywords, Ca24A_df)
print('\nBIOTIN (Gregor)')
cross_check(biotin_alt, biotin_keywords, Ca24A_df)
print('\nNIACIN (Gregor)')
cross_check(niacin, niacin_keywords, Ca24A_df)
print('\nNIACIN (Gregor alt.)')
cross_check(niacin_alt, niacin_keywords, Ca24A_df)
print('\nRIBOFLAVIN (Hessler)')
cross_check(riboflavin, riboflavin_keywords, Ca24A_df)
print('\nTHIAMINE (Hessler)')
cross_check(thiamine, thiamine_keywords, Ca24A_df)
print('\nTHIAMINE (Gregor)')
cross_check(thiamine_alt, thiamine_keywords, Ca24A_df)

BIOTIN (Hessler)
Step 1, bioCIW: 0 of 1 KOs found

Step 2, bioABDF: 4 of 3 KOs found
{'K00833', 'K00652', 'K01935', 'K01012'}

Found keywords ['bioA', 'bioB', 'bioD', 'bioF', 'biotin', '8-amino-7-oxononanoate']

BIOTIN (Gregor)
Step 1, bioF: 1 of 1 KOs found
{'K00652'}

Step 2, bioA: 1 of 1 KOs found
{'K00833'}

Step 3, bioD: 1 of 1 KOs found
{'K01935'}

Step 4, bioB: 1 of 1 KOs found
{'K01012'}


NIACIN (Gregor)
Step 1, nadC: 1 of 1 KOs found
{'K00767'}

Step 2, nadD: 1 of 1 KOs found
{'K00969'}

Step 3, nadE: 1 of 1 KOs found
{'K01950'}

Step 4, nadXB: 1 of 1 KOs found
{'K00278'}

Step 5, nadA: 1 of 1 KOs found
{'K03517'}


NIACIN (Gregor alt.)
Step 1, nadC: 1 of 1 KOs found
{'K00767'}

Step 2, nadD: 1 of 1 KOs found
{'K00969'}

Step 3, nadE: 1 of 1 KOs found
{'K01950'}

Step 4, kmo: 0 of 1 KOs found

Found keywords ['nadC', 'nadD', 'nadE', 'nadB', 'nadA', 'nicotin', 'NAD']

RIBOFLAVIN (Hessler)
Step 1, 1: 0 of 1 KOs found

Step 2, 2: 0 of 1 KOs found

Step 3, 3: 0 of 1 KOs found

St

In [10]:
# Pr24B_df = pd.read_csv('Pr24B_output_file.csv') # reload df from previous step if needed 
# Pr24B_df['KO'] = Pr24B_df['KO'].apply(literal_eval) # if reloading df from csv, turn string back into list

print('BIOTIN (Hessler)')
cross_check(biotin, biotin_keywords, Pr24B_df)
print('\nBIOTIN (Gregor)')
cross_check(biotin_alt, biotin_keywords, Pr24B_df)
print('\nNIACIN (Gregor)')
cross_check(niacin, niacin_keywords, Pr24B_df)
print('\nNIACIN (Gregor alt.)')
cross_check(niacin_alt, niacin_keywords, Pr24B_df)
print('\nRIBOFLAVIN (Hessler)')
cross_check(riboflavin, riboflavin_keywords, Pr24B_df)
print('\nTHIAMINE (Hessler)')
cross_check(thiamine, thiamine_keywords, Pr24B_df)
print('\nTHIAMINE (Gregor)')
cross_check(thiamine_alt, thiamine_keywords, Pr24B_df)

BIOTIN (Hessler)
Step 1, bioCIW: 1 of 1 KOs found
{'K02169'}

Step 2, bioABDF: 5 of 3 KOs found
{'K00652', 'K01012', 'K00833', 'K19563', 'K01935'}


BIOTIN (Gregor)
Step 1, bioF: 1 of 1 KOs found
{'K00652'}

Step 2, bioA: 2 of 1 KOs found
{'K00833', 'K19563'}

Step 3, bioD: 1 of 1 KOs found
{'K01935'}

Step 4, bioB: 1 of 1 KOs found
{'K01012'}


NIACIN (Gregor)
Step 1, nadC: 1 of 1 KOs found
{'K00767'}

Step 2, nadD: 1 of 1 KOs found
{'K00969'}

Step 3, nadE: 1 of 1 KOs found
{'K01916'}

Step 4, nadXB: 1 of 1 KOs found
{'K00278'}

Step 5, nadA: 1 of 1 KOs found
{'K03517'}


NIACIN (Gregor alt.)
Step 1, nadC: 1 of 1 KOs found
{'K00767'}

Step 2, nadD: 1 of 1 KOs found
{'K00969'}

Step 3, nadE: 1 of 1 KOs found
{'K01916'}

Step 4, kmo: 0 of 1 KOs found

Found keywords ['nadC', 'nadD', 'nadE', 'nadB', 'nadA', 'niacin', 'nicotin', 'NAD']

RIBOFLAVIN (Hessler)
Step 1, 1: 1 of 1 KOs found
{'K14652'}

Step 2, 2: 5 of 1 KOs found
{'K22912', 'K21063', 'K20861', 'K20862', 'K21064'}

Step 3, 3: 1

In [11]:
# Me12_df = pd.read_csv('Me12_output_file.csv') # reload df from previous step if needed 
# Me12_df['KO'] = Me12_df['KO'].apply(literal_eval) # if reloading df from csv, turn string back into list

print('BIOTIN (Hessler)')
cross_check(biotin, biotin_keywords, Me12_df)
print('\nBIOTIN (Gregor)')
cross_check(biotin_alt, biotin_keywords, Me12_df)
print('\nNIACIN (Gregor)')
cross_check(niacin, niacin_keywords, Me12_df)
print('\nNIACIN (Gregor alt.)')
cross_check(niacin_alt, niacin_keywords, Me12_df)
print('\nRIBOFLAVIN (Hessler)')
cross_check(riboflavin, riboflavin_keywords, Me12_df)
print('\nTHIAMINE (Hessler)')
cross_check(thiamine, thiamine_keywords, Me12_df)
print('\nTHIAMINE (Gregor)')
cross_check(thiamine_alt, thiamine_keywords, Me12_df)

BIOTIN (Hessler)
Step 1, bioCIW: 0 of 1 KOs found

Step 2, bioABDF: 0 of 3 KOs found

Found keywords ['bioF', 'biotin']

BIOTIN (Gregor)
Step 1, bioF: 0 of 1 KOs found

Step 2, bioA: 0 of 1 KOs found

Step 3, bioD: 0 of 1 KOs found

Step 4, bioB: 0 of 1 KOs found

Found keywords ['bioF', 'biotin']

NIACIN (Gregor)
Step 1, nadC: 0 of 1 KOs found

Step 2, nadD: 1 of 1 KOs found
{'K00969'}

Step 3, nadE: 2 of 1 KOs found
{'K01916', 'K01950'}

Step 4, nadXB: 0 of 1 KOs found

Step 5, nadA: 0 of 1 KOs found

Found keywords ['nadD', 'nadE', 'nicotin', 'NAD']

NIACIN (Gregor alt.)
Step 1, nadC: 0 of 1 KOs found

Step 2, nadD: 1 of 1 KOs found
{'K00969'}

Step 3, nadE: 2 of 1 KOs found
{'K01916', 'K01950'}

Step 4, kmo: 0 of 1 KOs found

Found keywords ['nadD', 'nadE', 'nicotin', 'NAD']

RIBOFLAVIN (Hessler)
Step 1, 1: 1 of 1 KOs found
{'K14652'}

Step 2, 2: 0 of 1 KOs found

Step 3, 3: 0 of 1 KOs found

Step 4, 4: 1 of 1 KOs found
{'K11752'}

Step 5, 5: 0 of 1 KOs found

Step 6, 6: 1 of 1 KOs